# Lecture 2: Computational Thinking with Python

## Exercise 2: Chemical Equation Balancer Checker

**Objective:** Write a program that checks if a chemical equation is balanced.

**Problem:** Create a function called `is_balanced()` that takes a chemical equation string and returns True if balanced, False if not.

**Your Task:**
- Split the equation at the arrow symbol "→" to get reactants and products
- For each side, parse the molecules and their coefficients
- Count total atoms of each element on both sides
- Compare the totals

Example Equation Format: "2H2 + O2 → 2H2O"

**Hints:**

- Use equation.split("→") to separate reactants from products
- Use side.split(" + ") to separate individual molecules
- You'll need to handle coefficients (numbers in front of molecules)
- Reuse your count_atoms() function from Exercise 1!
- For molecules with coefficients like "2H2O", the coefficient applies to all atoms in that molecule

**Steps to Think Through:**

- How do you extract the coefficient from "2H2O"? (It's everything before the first letter)
- How do you multiply the atom counts by the coefficient?
- How do you combine counts from multiple molecules on the same side?


In [3]:
from typing import Dict

def parse_molecular_formula(formula: str) -> Dict[str, int]:
    atom_dict = dict()

    for i in range(len(formula)):

        # Check if it is a element bc they start with a big letter, otherwise: nothing
        if formula[i].isupper() == True:

            element = formula[i]

            # Check if it is a two-letter element, (and stop before formula is done to not get an error)
            if i + 1 < len(formula) and formula[i + 1].islower() ==True:
                element = formula[i:i + 2]

                # Check for a number after the two-letter element, (and stop before formula is done to not get an error)
                if i + 2 < len(formula) and formula[i + 2].isdigit() == True:

                    number = formula[i + 2]

                    # Check for a number after the number, (and stop before formula is done to not get an error)
                    if i + 3 < len(formula) and formula[i + 3].isdigit() == True:
                        number += formula[i + 3]

                    count = int(number) # turn the count into an integer to perform math. activities with it

                else:
                    count = 1

            # If no small letter is found
            else:

                # Check for a number after the element (and stop before formula is done to not get an error)
                if i + 1 < len(formula) and formula[i + 1].isdigit() == True:

                    number = formula[i + 1]

                    # Check for a second digit (and stop before formula is done to not get an error)
                    if i + 2 < len(formula) and formula[i + 2].isdigit() == True:
                        number += formula[i + 2]

                    count = int(number)

                else:
                    count = 1

            # Add element to dictionary
            if element in atom_dict:
                atom_dict[element] += count
            else:
                atom_dict[element] = count
        else:
            continue

    return atom_dict

In [ ]:
def count_side(side: str) -> Dict[str, int]:
    total_atoms = {}

    #split the side into molecules
    molecules = side.split("+")

    for molecule in molecules:

        molecule = molecule.strip() #removes spaces (bc i got an error without this, i think bc i use len() and space counts as a character)        
        coefficient_text = "" #coefficient at the beginning
        i = 0

        while i < len(molecule) and molecule[i].isdigit():
            coefficient_text += molecule[i]
            i += 1
        
        if coefficient_text == "": #if there is no coefficient, coefficient = 1
            coefficient = 1
        else:
            coefficient = int(coefficient_text)

        #everything after the coefficient is formula
        formula = molecule[i:]

        molecule_atoms = parse_molecular_formula(formula) #count the atoms in this molecule

        for element, count in molecule_atoms.items(): #multipling atoms by the coefficient

            total_count = count * coefficient

            if element in total_atoms:
                total_atoms[element] += total_count
            else:
                total_atoms[element] = total_count

    return total_atoms


def is_balanced(equation: str) -> bool:
    
    if "→" not in equation: #make sure the equation contains an arrow
        return False

    reactants, products = equation.split("→", 1) #split into reactants and products

    reactant_atoms = count_side(reactants) #count all atoms on each side
    product_atoms = count_side(products)

    return reactant_atoms == product_atoms #compare dictionaries

### Test Cases


In [5]:
# If the outcome is incorrect, the keyword assert will cause an error

print(is_balanced("2H2 + O2 → 2H2O"))
assert is_balanced("2H2 + O2 → 2H2O")

print(is_balanced("H2 + O2 → H2O"))
assert not is_balanced("H2 + O2 → H2O")

print(is_balanced("CH4 + 2O2 → CO2 + 2H2O"))
assert is_balanced("CH4 + 2O2 → CO2 + 2H2O")

print(is_balanced("N2 + H2 → NH3"))
assert not is_balanced("N2 + H2 → NH3")

True
False
True
False


## Exercise 3: Reaction Pathway Analyzer

**Objective:** Analyze multi-step reaction sequences to find synthesis pathways.

**Problem:** Write a program that determines what compounds can be synthesized from starting materials through a series of reactions, and finds a pathway to create a target compound.

**Your Task:**
Create a function called `find_synthesis_path()` that takes:

- A list of starting compounds
- A list of reaction equations
- A target compound to synthesize

The function should return either:

- A list of reaction steps that lead to the target compound, OR
- A message saying the target cannot be synthesized

**Hints:**

- Start by creating a set of "available" compounds (your starting materials)
- For each reaction, check if you have all the reactants needed
- If you can perform a reaction, add the products to your available compounds
- Keep track of which reactions you've used in what order
- Continue until you either find the target or can't make any new compounds

#### Computational Thinking:

**Decomposition:** Break this into smaller problems

- Parse reaction equations
- Track available compounds
- Check if reactions can be performed
- Record the synthesis pathway


**Pattern Recognition:** What patterns do you see?

- Each reaction has the same format: "reactants → products"
- You need to repeatedly check if new reactions become possible


**Algorithm Design:** What's your step-by-step approach?

- Initialize available compounds
- Loop: try each reaction, see if possible, update available compounds
- Stop when target is found or no progress is made


**Abstraction:** What functions might be helpful?

- A function to parse a single reaction equation
- A function to check if a reaction is possible with current compounds
- A function to update available compounds after a reaction




In [13]:
from typing import List

def find_synthesis_path(starting_materials: List[str], reactions: List[str], target: str) -> List[str] | str:

    compounds1= set(starting_materials) #compounds available

    pathways = {}  #store the pathway needed to make each compound

    for material in starting_materials:
        pathways[material] = []

    if target in compounds1:#if we already have the target
        return []

    while True: #keep checking reactions until no new compounds 

        progress = False

        for reaction in reactions:
            reactant_side, product_side = reaction.split("→") #split reaction into reactants and products

            reactants = reactant_side.split("+")
            products = product_side.split("+")

            clean_reactants = []
            clean_products = []

            for reactant in reactants: #remove spaces and coefficients from reactants

                reactant = reactant.strip()

                i = 0
                while i < len(reactant) and reactant[i].isdigit():
                    i += 1

                reactant = reactant[i:]
                clean_reactants.append(reactant)
            
            for product in products: #remove spaces and coefficients

                product = product.strip()

                i = 0
                while i < len(product) and product[i].isdigit():
                    i += 1

                product = product[i:]
                clean_products.append(product)

            can_perform = True #check whether all reactants are available

            for reactant in clean_reactants:
                if reactant not in compounds1:
                    can_perform = False
                    break

            if can_perform: #if we can perform the reaction

                current_path = []  #pathway needed for this reaction

                for reactant in clean_reactants:

                    for step in pathways[reactant]:
                        if step not in current_path:
                            current_path.append(step)

                current_path.append(reaction)

                for product in clean_products: #Add the new products

                    if product not in compounds1:
                        compounds1.add(product)
                        pathways[product] = current_path.copy()

                        progress = True

                        if product == target:  #target found
                            return pathways[product]
                        
        if progress == False:  #if no new compounds were produced, stop
            return "Target cannot be synthesized"

### Test Cases

In [14]:
# Basic Multi-Step Synthesis

starting_materials = ["H2", "O2", "N2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3", 
    "NH3 + H2O → NH4OH"
]
target = "NH4OH"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find a path using H2 + N2 → NH3, then NH3 + H2O → NH4OH

['N2 + 3H2 → 2NH3', '2H2 + O2 → 2H2O', 'NH3 + H2O → NH4OH']


In [9]:
# Direct Synthesis (One Step)

starting_materials = ["Na", "Cl2"]
reactions = [
    "2Na + Cl2 → 2NaCl",
    "NaCl + H2O → NaOH + HCl"
]
target = "NaCl"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find direct synthesis in one step

['2Na + Cl2 → 2NaCl']


In [10]:
# Impossible Synthesis

starting_materials = ["H2", "O2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3"  # But we don't have N2!
]
target = "NH3"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should return message that NH3 cannot be synthesized

Target cannot be synthesized


In [11]:
# Multiple Pathway Options
starting_materials = ["C", "H2", "O2", "H2O"]
reactions = [
    "C + O2 → CO2",
    "2H2 + O2 → 2H2O",
    "CO2 + H2O → H2CO3",
    "C + 2H2 → CH4",
    "CH4 + 2O2 → CO2 + 2H2O"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find one of multiple possible paths to CO2

['C + O2 → CO2']


In [12]:
# Long Chain Synthesis
starting_materials = ["Fe", "O2", "C"]
reactions = [
    "4Fe + 3O2 → 2Fe2O3",
    "2C + O2 → 2CO",
    "Fe2O3 + 3CO → 2Fe + 3CO2",
    "CO2 + C → 2CO"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find the multi-step path involving iron oxide formation and reduction

['4Fe + 3O2 → 2Fe2O3', '2C + O2 → 2CO', 'Fe2O3 + 3CO → 2Fe + 3CO2']
